# Stage 17: Advanced Visualization & Power BI Layer
**Project**: Cyber Crime Analytics for National Security  
**Syllabus Alignment**: Comprehensive Project Synthesis & Visualization  
**Deliverables**: 10-Page Power BI Architecture, 28-Table Semantic Data Package, 25+ DAX Measures, and Validation Gate  
**Data Sources**: Validated 2023 Cross-Sectional Facts (`data/processed/master_state_2023.csv`) & Historical 2018–2022 Panel Facts (`data/processed/trend_2018_2022.csv`)  

---

## 1. Executive Architecture & Purpose

Stage 17 integrates all analytical findings from **Stages 1 through 16** into a unified, transparent, and academically rigorous **10-Page Visual Analytics Suite**.

```
========================================================================================
                      10-PAGE POWER BI DASHBOARD ARCHITECTURE
========================================================================================
 [Page 1: Executive Overview] ──────► National KPIs (86,420 cases), Act Groups & Pareto
 [Page 2: Geographic Analysis] ─────► 36 State/UT Rankings, Top 5 Concentration (73.45%)
 [Page 3: Categories & Motives] ────► 40 Leaf Offenses, 18 Motives, Zero Double-Counting
 [Page 4: Association Mining] ──────► FP-Growth & Apriori Rules (State-Level Demo)
 [Page 5: Classification] ──────────► Regime Classification (DT, NB, SVM, RF on 2022 Test)
 [Page 6: Regression & Forecasting] ► 1-Year Lag Forecasting (Log-Linear OLS R²=0.9000)
 [Page 7: Cluster Profiling] ───────► K=4 Structural Profiles, Ward & DBSCAN Validation
 [Page 8: Anomaly & Outlier Study] ─► Multi-Method Consensus (0–4), Dual-Space Analysis
 [Page 9: Historical Panel Trends] ─► 2018–2022 Growth Trajectories (Isolated Series)
 [Page 10: Methodology & Limits] ───► Star Schema Provenance, 6 Core Limitations & Guardrails
========================================================================================
```

> **Runtime Environment Disclosure**: Power BI Desktop is a Windows desktop GUI application and is not executable via CLI in this headless runtime. In strict compliance with non-fabrication rules, no synthetic binary `.pbix` is generated. All underlying data, measures, models, layouts, and relationships are fully specified, verified, and reconciled against the master analytical database.

In [1]:
# Environment setup and package imports
import os
import sys
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

POWERBI_DATA = project_root / 'dashboard' / 'powerbi_data'
print(f"[+] Advanced Visualization environment initialized. Power BI Data Path: {POWERBI_DATA}")

[+] Advanced Visualization environment initialized. Power BI Data Path: C:\Users\kaval vyas\OneDrive\Desktop\Projects\cybercrime-analytics\dashboard\powerbi_data


## 2. Page 1: Executive Overview & National 2023 KPIs

We load the headline executive KPI table and verify national totals across legal frameworks, primary motives, and demographic subsets.

In [2]:
kpi_df = pd.read_csv(POWERBI_DATA / 'kpi_executive_summary.csv')
print("=== Page 1: Executive KPI Summary ===")
display(kpi_df)

act_df = pd.read_csv(POWERBI_DATA / 'dim_act_group.csv')
print("\n=== Page 1: Act Group Distribution ===")
display(act_df)

=== Page 1: Executive KPI Summary ===

=== Page 1: Act Group Distribution ===


,kpi_name,kpi_value_num,kpi_value_str,kpi_unit,category,source_stage
0,Total 2023 Cybercrime Cases,86420.00,"86,420",Cases,Volume,Stage 3 & 4
1,Total States & UTs Evaluated,36.00,36,Jurisdictions,Scope,Stage 2
2,IT Act Cases,44237.00,"44,237",Cases,Legal Framework,Stage 3
3,IT Act Share,51.19,51.19%,Percentage,Legal Framework,Stage 3
4,IPC Crimes r/w IT Act Cases,41849.00,"41,849",Cases,Legal Framework,Stage 3
5,IPC Crimes r/w IT Act Share,48.43,48.43%,Percentage,Legal Framework,Stage 3
6,Special & Local Laws (SLL) Cases,334.00,334,Cases,Legal Framework,Stage 3
7,Fraud Motive Cases,59526.00,"59,526",Cases,Motive,Stage 3 & 4
8,Fraud Motive Share,68.88,68.88%,Percentage,Motive,Stage 3 & 4
9,Cybercrimes Against Women,19510.00,"19,510",Cases,Demographic Subset,Stage 4


,act_group,leaf_category_count,total_cases,share_pct
0,IT Act,18,44237,51.19
1,IPC,17,41849,48.43
2,SLL,5,334,0.39


## 3. Page 2: Geographic & State Analysis (Top 5 Concentration)

We evaluate cross-sectional distributions across 36 States/UTs, highlighting that the Top 5 volume jurisdictions account for $73.45\%$ ($63,472$ cases) of all registered cybercrimes.

In [3]:
state_df = pd.read_csv(POWERBI_DATA / 'state_summary_2023.csv')
top5 = state_df.sort_values(by='total_cases', ascending=False).head(5)
top5['share_pct'] = (top5['total_cases'] / 86420 * 100).round(2)
print("=== Page 2: Top 5 High-Volume Jurisdictions ===")
display(top5[['state_name', 'total_cases', 'share_pct', 'it_act_cases', 'ipc_cases', 'motive_fraud']])

=== Page 2: Top 5 High-Volume Jurisdictions ===


,state_name,total_cases,share_pct,it_act_cases,ipc_cases,motive_fraud
0,Karnataka,21889,25.33,21870,18,18602
1,Telangana,18236,21.10,439,17787,14749
2,Uttar Pradesh,10794,12.49,10102,689,3811
3,Maharashtra,8103,9.38,960,7131,5290
4,Bihar,4450,5.15,397,4052,3422


## 4. Page 3: Offense Taxonomy & Motive Hierarchy

We inspect the 40 independent leaf categories and 18 specific motives, ensuring zero double-counting of parent subtotals.

In [4]:
cat_df = pd.read_csv(POWERBI_DATA / 'category_summary_2023.csv')
motive_df = pd.read_csv(POWERBI_DATA / 'motive_summary_2023.csv')

print("=== Page 3: Top 10 Leaf Categories by Volume ===")
display(cat_df[['category_rank', 'category_display_name', 'act_group', 'national_cases', 'national_share_pct', 'cumulative_share_pct']].head(10))

print("\n=== Page 3: Top 5 Primary Motives ===")
display(motive_df.head(5))

=== Page 3: Top 10 Leaf Categories by Volume ===

=== Page 3: Top 5 Primary Motives ===


,category_rank,category_display_name,act_group,national_cases,national_share_pct,cumulative_share_pct
0,1,Computer Related Offences - D) Cheating by personation by using computer resource (Sec.66D),IT Act,25334,29.31,29.31
1,2,Cheating (Sec.420 IPC),IPC,16943,19.61,48.92
2,3,"Fraud (Sec.420 r/w Sec.465, 468- 471 IPC) - E) Others",IPC,6034,6.98,55.90
3,4,"Fraud (Sec.420 r/w Sec.465, 468- 471 IPC) - D) OTP Frauds",IPC,5116,5.92,61.82
4,5,Computer Related Offences - C) Identity Theft (Sec.66C),IT Act,4978,5.76,67.58
5,6,"Fraud (Sec.420 r/w Sec.465, 468- 471 IPC) - C) Online Banking Fraud",IPC,4435,5.13,72.71
6,7,Computer Related Offences - Computer Related Offences (Sec.66) - a2) Offences other than Ransom-ware,IT Act,3359,3.89,76.60
7,8,Publication/ transmission of obscene / sexually explicit act in electronic form (Sec. 67) - A) Publishing or transmitting obscene material in Electronic Form,IT Act,3110,3.60,80.20
8,9,Other Offences (r/w IT Act),IPC,2389,2.76,82.96
9,10,Publication/transmission of obscene / sexually explicit act in electronic form (Sec. 67) - B) Publishing or transmitting of material containing Sexually explicit act in electronic form (Sec.67A),IT Act,2168,2.51,85.47


,motive_id,motive_display_name,is_total,national_motive_count,states_reporting,share_pct
0,3,Fraud,0,59526,33,68.88
1,18,Others,0,13025,31,15.07
2,7,Sexual Exploitation,0,4199,31,4.86
3,4,Extortion,0,3326,28,3.85
4,5,Causing Disrepute,0,2436,25,2.82


## 5. Page 4 & 5: Pattern Mining & Supervised Classification Packages

We verify the Stage 5/12 association rule mining extracts and Stage 13 supervised classification leaderboard on the held-out 2022 test set.

In [5]:
rules_df = pd.read_csv(POWERBI_DATA / 'model_association_rules_key.csv')
print("=== Page 4: Top Association Rules (State-Level Demo) ===")
display(rules_df[['rule_id', 'antecedent', 'consequent', 'support', 'confidence', 'lift']].head(6))

cls_df = pd.read_csv(POWERBI_DATA / 'model_classification_comparison.csv')
cm_df = pd.read_csv(POWERBI_DATA / 'model_classification_confusion_matrices.csv')
print("\n=== Page 5: Supervised Classification Leaderboard (Held-Out 2022 Test) ===")
display(cls_df)
print("\n=== Page 5: Confusion Matrices ===")
display(cm_df)

=== Page 4: Top Association Rules (State-Level Demo) ===

=== Page 5: Supervised Classification Leaderboard (Held-Out 2022 Test) ===

=== Page 5: Confusion Matrices ===


,rule_id,antecedent,consequent,support,confidence,lift
0,R-01,HIGH_SEXUAL_EXPLOITATION_MOTIVE,HIGH_EXTORTION_MOTIVE,0.472222,0.944444,1.888889
1,R-02,HIGH_EXTORTION_MOTIVE,HIGH_SEXUAL_EXPLOITATION_MOTIVE,0.472222,0.944444,1.888889
2,R-03,HIGH_EXTORTION_MOTIVE,HIGH_IDENTITY_THEFT,0.472222,0.944444,1.789474
3,R-04,HIGH_IDENTITY_THEFT,HIGH_EXTORTION_MOTIVE,0.472222,0.894737,1.789474
4,R-05,HIGH_FRAUD_MOTIVE,HIGH_EXTORTION_MOTIVE,0.444444,0.888889,1.777778
5,R-06,HIGH_EXTORTION_MOTIVE,HIGH_FRAUD_MOTIVE,0.444444,0.888889,1.777778


,Model,Accuracy,Balanced_Accuracy,Precision,Recall,Specificity,F1_Score,ROC_AUC
0,Baseline (Most Frequent),0.4444,0.500,0.0,0.00,1.0,0.0000,0.500
1,Decision Tree (depth=3),0.9722,0.975,1.0,0.95,1.0,0.9744,0.975
2,Gaussian Naive Bayes,1.0000,1.000,1.0,1.00,1.0,1.0000,1.000
3,Linear SVM,1.0000,1.000,1.0,1.00,1.0,1.0000,1.000
4,RBF SVM,1.0000,1.000,1.0,1.00,1.0,1.0000,1.000
5,Random Forest,1.0000,1.000,1.0,1.00,1.0,1.0000,1.000


,Model,True_Negative_TN,False_Positive_FP,False_Negative_FN,True_Positive_TP,Total_Test_Obs
0,Baseline (Most Frequent),16,0,20,0,36
1,Decision Tree (depth=3),16,0,1,19,36
2,Gaussian Naive Bayes,16,0,0,20,36
3,Linear SVM,16,0,0,20,36
4,RBF SVM,16,0,0,20,36
5,Random Forest,16,0,0,20,36


## 6. Page 6 & 7: Regression Forecasting & Cluster Profiling Packages

We inspect the continuous forecasting leaderboard (Log-Linear OLS $R^2 = 0.9000$) and the K=4 structural cluster composition profiles.

In [6]:
reg_df = pd.read_csv(POWERBI_DATA / 'model_prediction_metrics.csv')
print("=== Page 6: Predictive Regression Leaderboard (2022 Evaluation) ===")
display(reg_df[['model_name', 'model_type', 'mae', 'rmse', 'r2', 'median_ae']])

clust_prof = pd.read_csv(POWERBI_DATA / 'model_cluster_profiles.csv')
print("\n=== Page 7: K=4 Cluster Composition Profiles ===")
display(clust_prof)

=== Page 6: Predictive Regression Leaderboard (2022 Evaluation) ===

=== Page 7: K=4 Cluster Composition Profiles ===


,model_name,model_type,mae,rmse,r2,median_ae
0,Naive Persistent (Lag-1),Baseline,564.75,1340.75,0.8625,61.50
1,Historical 2-Year Moving Average,Baseline,563.04,1523.73,0.8224,57.00
2,"Linear Regression (OLS, Raw)",Linear Model,776.08,1680.13,0.7840,205.07
3,Ridge Regression (L2 Regularized),Linear Model,776.08,1680.13,0.7840,205.07
4,Log-Linear Regression (Log OLS),Log-Linear Model,479.37,1143.46,0.9000,69.85
5,Decision Tree Regressor,Tree Model,608.42,1571.47,0.8111,100.86
6,Random Forest Regressor,Ensemble Model,582.24,1559.55,0.8139,58.47


,cluster_id,cluster_label,state_count,state_pct,it_act_share_mean,it_act_share_median,it_act_share_std,fraud_motive_share_mean,fraud_motive_share_median,fraud_motive_share_std,extortion_motive_share_mean,extortion_motive_share_median,extortion_motive_share_std,sexual_exploitation_motive_share_mean,sexual_exploitation_motive_share_median,sexual_exploitation_motive_share_std
0,0,Lower IT Act Share / Moderate Fraud Share Profile,15,41.67,0.3267,0.3145,0.2340,0.4442,0.4681,0.2556,0.0276,0.0248,0.0259,0.1103,0.1021,0.1022
1,1,Higher IT Act Share / Higher Fraud Share Profile,12,33.33,0.8868,0.9445,0.1628,0.7161,0.7414,0.1730,0.0194,0.0165,0.0187,0.1053,0.0692,0.1119
2,2,High Sexual-Exploitation Share / Small-Denominator Profile,2,5.56,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.9167,0.9167,0.1179
3,3,Higher Extortion Motive Share Profile,7,19.44,0.7440,0.8441,0.2644,0.3325,0.3531,0.1645,0.1254,0.1304,0.0336,0.1438,0.1448,0.1300


## 7. Page 8: Advanced Outlier & Anomaly Validation

We examine multi-method consensus scoring ($0–4$), robust Mahalanobis distances ($\chi^2_{14, 0.975} = 26.12$ reference screening threshold), and dual-space volume vs. composition separation.

In [7]:
anom_df = pd.read_csv(POWERBI_DATA / 'model_outlier_consensus.csv')
print("=== Page 8: Consensus Anomaly Breakdown (Score >= 2) ===")
display(anom_df[anom_df['consensus_score'] >= 2][['state_name', 'total_cases', 'consensus_score', 'consensus_classification']])

vol_comp = pd.read_csv(POWERBI_DATA / 'model_outlier_volume_vs_composition.csv')
print("\n=== Page 8: Dual-Space Anomaly Orientation Summary ===")
display(vol_comp['anomaly_orientation'].value_counts())

=== Page 8: Consensus Anomaly Breakdown (Score >= 2) ===

=== Page 8: Dual-Space Anomaly Orientation Summary ===


,state_name,total_cases,consensus_score,consensus_classification
0,Karnataka,21889,4,4 (Consensus anomaly)
1,Dadra and Nagar Haveli and Daman and Diu,6,4,4 (Consensus anomaly)
2,Lakshadweep,1,4,4 (Consensus anomaly)
3,Uttar Pradesh,10794,3,3 (Strong multi-method anomaly)
4,Jharkhand,1079,3,3 (Strong multi-method anomaly)
5,Ladakh,1,3,3 (Strong multi-method anomaly)
6,Telangana,18236,2,2 (Multi-method anomaly)
7,Kerala,3295,2,2 (Multi-method anomaly)
8,Odisha,2348,2,2 (Multi-method anomaly)


anomaly_orientation
Not Anomalous in Tested Spaces        17
Composition-Profile Driven Anomaly     8
Both Volume & Composition Anomaly      6
Volume-Scale Driven Anomaly            5
Name: count, dtype: int64


## 8. Page 9 & 10: Historical Trends & Core Academic Limitations

We verify the 2018–2022 historical growth series ($27,248 \to 65,893$) and display all 6 core project limitations.

In [8]:
trend_nat = pd.read_csv(POWERBI_DATA / 'trend_national_2018_2022.csv')
print("=== Page 9: 5-Year National Growth Trajectory (2018–2022) ===")
display(trend_nat)

limits_df = pd.read_csv(POWERBI_DATA / 'metadata_project_limitations.csv')
print("\n=== Page 10: 6 Core Academic Limitations ===")
for _, r in limits_df.iterrows():
    print(f"[{r['limitation_id']}] {r['title']} ({r['scope']}):")
    print(f"    {r['description']}\n")

=== Page 9: 5-Year National Growth Trajectory (2018–2022) ===

=== Page 10: 6 Core Academic Limitations ===
[LIM-01] Absence of Population Normalization (Geographic & Volume Comparisons):
    All figures represent raw reported police case counts from NCRB Table 9A.1. No state population data are included; counts cannot be interpreted as per-capita incidence rates or crime risk scores.

[LIM-02] Strict Time-Series Discontinuity (2018–2022 vs 2023) (Temporal Trend Analysis):
    Historical 2018–2022 series (Rajya Sabha unstarred question / NCRB archival) and 2023 detailed dataset originate from distinct recording tables with discrepancies. They are maintained as separate series and must never be concatenated into a continuous 2018–2023 line.

[LIM-03] Short Historical Horizon (Time Series & Forecasting):
    Only 5 historical annual cross-sections exist nationally. Classical ARIMA/time-series forecasting is statistically unjustifiable; longitudinal analysis is restricted to 1-year panel 

,year,national_total_cases,prev_year_cases,yoy_growth_pct,series_note
0,2018,27248.0,NaN,NaN,Historical NCRB/Rajya Sabha Series (2018-2022) - Maintained separately from 2023 NCRB dataset
1,2019,44735.0,27248.0,64.18,Historical NCRB/Rajya Sabha Series (2018-2022) - Maintained separately from 2023 NCRB dataset
2,2020,50035.0,44735.0,11.85,Historical NCRB/Rajya Sabha Series (2018-2022) - Maintained separately from 2023 NCRB dataset
3,2021,52974.0,50035.0,5.87,Historical NCRB/Rajya Sabha Series (2018-2022) - Maintained separately from 2023 NCRB dataset
4,2022,65893.0,52974.0,24.39,Historical NCRB/Rajya Sabha Series (2018-2022) - Maintained separately from 2023 NCRB dataset


## 9. Stage 17 Quality Gate & Validation Execution

We execute the formal Stage 17 automated validation suite (`src/validate_stage17.py`) to confirm 100% mathematical reconciliation and structural integrity.

In [9]:
from src.validate_stage17 import validate_stage17
success = validate_stage17()
print(f"\n[+] Stage 17 Validation Status: {'PASSED (100%)' if success else 'FAILED'}")

RUNNING STAGE 17 VALIDATION: Advanced Visualization & Power BI Layer
[1] File Integrity: All 28 Power BI data tables and documentation exist: PASSED
[2] Dimensional & Fact Structure (36 states, 49 categories, 19 motives, 3 acts): PASSED
[3] National 2023 KPI Reconciliation (86,420 total, 44,237 IT, 41,849 IPC, 334 SLL): PASSED
[4] Geographic Ranking & Top 5 Concentration (63,472 / 73.45%): PASSED
[5] Crime Category Offenses & Motives (Sec 66D: 25,334, Sec 420: 16,943, Fraud: 59,526): PASSED
[6] Association Rule Mining Package (42 rules, validated benchmarks): PASSED
[7] Supervised Classification Package (Stage 13 DT=97.22%, RF=100.0%): PASSED
[8] Predictive Regression Package (Stage 7/14 Log-OLS MAE=479.37, R²=0.9000): PASSED
[9] Unsupervised Cluster Profiling Package (K=4 sizes: 15, 12, 2, 7): PASSED
[10] Advanced Outlier & Anomaly Package (Consensus 0-4, Mahalanobis, LOF, Dual-Space): PASSED
[11] Historical Series Isolation (2018-2022 panel, 27,248 to 65,893 cases): PASSED
[12] Speci